# BT4012 Competition — RF + XGBoost Ensemble

Blends the final XGBoost recipe from `03_supervised_learning.ipynb` with Random Forest, to test whether combining the two improves on XGBoost alone. Scored by ROC-AUC, the competition metric.

**Feature set:** this experiment was run when the best submission used 79 features (degree and centrality graph features, before the community and neighborhood features were added), so it keeps that feature set and excludes the six later graph features. The XGBoost-alone predictions reproduce that 79-feature submission exactly.

**Models:**
- **XGBoost**: recency-weighted (`half_life=10`) with the tuned hyperparameters, identical to `final_model`.
- **Random Forest**: 300 trees, `class_weight="balanced"`, the baseline from `03_supervised_learning.ipynb`.

**Blending:** each prediction is `w × XGBoost + (1 − w) × RF`, using either:
- **probability average**: the raw predicted probabilities, or
- **rank average**: each model's predictions converted to percentile ranks first. ROC-AUC only depends on how transactions are ordered, and RF's vote fractions and XGBoost's probabilities sit on different scales, so rank averaging usually blends them more evenly.

**Caveat:** on the 31–35 validation window every model scores ROC-AUC ≈ 0.997–0.998, close to the ceiling, so validation can show whether blending clearly hurts but not reliably whether it helps. The leaderboard is the real test.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

## Data

In [2]:
train_reduced = pd.read_csv("data/processed/train_reduced.csv")
test_reduced = pd.read_csv("data/processed/test_reduced.csv")

EXCLUDE = {
    "txId", "time_step", "label",
    "label_propagated", "label_propagated_confidence",                  # pseudo-label columns, not features
    "community_size", "community_density", "community_internal_ratio",  # added after this experiment
    "flow_ratio", "mean_neighbor_degree", "hop2_reach",                # added after this experiment
}
feat_cols = [c for c in train_reduced.columns if c not in EXCLUDE]

labeled = train_reduced[train_reduced["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

SPLIT_POINT = 30
train_df = labeled[labeled["time_step"] <= SPLIT_POINT]
val_df = labeled[labeled["time_step"] > SPLIT_POINT]

print("features:", len(feat_cols))
print(f"train: {len(train_df)} rows, validation (time_step 31-35): {len(val_df)} rows, test: {len(test_reduced)} rows")

features: 79
train: 26905 rows, validation (time_step 31-35): 4330 rows, test: 15329 rows


## Models

`fit_models` trains both models on whatever rows it's given. XGBoost's recency weight is anchored at the latest `time_step` in those rows: 30 for the validation fit, 35 for the final fit.

In [3]:
BEST_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=1.0,
                   colsample_bytree=0.7, reg_lambda=5, eval_metric="logloss", random_state=42, n_jobs=-1)
HALF_LIFE = 10


def xgb_sample_weight(rows):
    y = rows["label"]
    class_ratio = (y == 0).sum() / (y == 1).sum()
    recency = 0.5 ** ((rows["time_step"].max() - rows["time_step"]) / HALF_LIFE)
    return y.map({0: 1.0, 1: class_ratio}) * recency


def fit_models(rows):
    xgb = XGBClassifier(**BEST_PARAMS)
    xgb.fit(rows[feat_cols], rows["label"], sample_weight=xgb_sample_weight(rows))
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(rows[feat_cols], rows["label"])
    return xgb, rf


def blend(p_xgb, p_rf, w_xgb, method):
    if method == "rank":
        p_xgb = pd.Series(p_xgb).rank(pct=True).to_numpy()
        p_rf = pd.Series(p_rf).rank(pct=True).to_numpy()
    return w_xgb * p_xgb + (1 - w_xgb) * p_rf

## Validation (train ≤ 30, validate on 31–35)

How differently the two models rank transactions matters most: a blend can only help if they disagree. A Spearman correlation near 1 means they order transactions almost identically, leaving little for a blend to gain.

In [4]:
xgb_val, rf_val = fit_models(train_df)
p_xgb = xgb_val.predict_proba(val_df[feat_cols])[:, 1]
p_rf = rf_val.predict_proba(val_df[feat_cols])[:, 1]

print(f"Spearman correlation between XGBoost and RF predictions: {spearmanr(p_xgb, p_rf).correlation:.4f}")

rows = []
for w in [1.0, 0.75, 0.5, 0.25, 0.0]:
    rows.append({
        "xgb_weight": w,
        "probability average": roc_auc_score(val_df["label"], blend(p_xgb, p_rf, w, "prob")),
        "rank average": roc_auc_score(val_df["label"], blend(p_xgb, p_rf, w, "rank")),
    })
pd.DataFrame(rows).round(4)

Spearman correlation between XGBoost and RF predictions: 0.8260


,xgb_weight,probability average,rank average
0,1.00,0.9973,0.9973
1,0.75,0.9977,0.9978
2,0.50,0.9979,0.9980
3,0.25,0.9979,0.9979
4,0.00,0.9975,0.9975


## Final models and submissions

Both models are retrained on all 31,235 labeled rows. The XGBoost-alone predictions should match `data/submissions/experiments/submission_centrality_79feat.csv` exactly, which confirms this notebook uses the same features and recipe as the best submission.

Two rank-averaged blends are exported: 50/50 and 70/30 XGBoost/RF. Rank-averaged scores are percentiles between 0 and 1 rather than calibrated probabilities, which is fine for ROC-AUC since only the ordering is scored. `data/submissions/final_submission.csv` is not touched.

In [5]:
xgb_full, rf_full = fit_models(labeled)
t_xgb = xgb_full.predict_proba(test_reduced[feat_cols])[:, 1]
t_rf = rf_full.predict_proba(test_reduced[feat_cols])[:, 1]

reference = pd.read_csv("data/submissions/experiments/submission_centrality_79feat.csv")["target"].to_numpy()
print("XGBoost alone matches submission_centrality_79feat.csv:", np.allclose(t_xgb, reference, atol=1e-6))

for w, filename in [(0.5, "submission_ensemble_rank50.csv"), (0.7, "submission_ensemble_rank70.csv")]:
    target = blend(t_xgb, t_rf, w, "rank")
    pd.DataFrame({"index": np.arange(len(test_reduced)), "target": target}).to_csv(f"data/submissions/experiments/{filename}", index=False)
    print(f"saved data/submissions/experiments/{filename} (XGBoost weight {w})")

XGBoost alone matches submission_centrality_79feat.csv: True
saved data/submissions/experiments/submission_ensemble_rank50.csv (XGBoost weight 0.5)
saved data/submissions/experiments/submission_ensemble_rank70.csv (XGBoost weight 0.7)


## Outcome

Random Forest ranks transactions somewhat differently from XGBoost (Spearman correlation 0.83 on validation), and the rank-averaged blends score slightly higher than XGBoost alone on the 31–35 window. On the leaderboard, however, the blend scored below XGBoost alone, so the final model (`03_supervised_learning.ipynb`, section 4) uses XGBoost only. This window sits right after training and is close to its ceiling, so it can't reliably show whether a blend helps on the later test period.
